
# 01 一人量化公司：主体合规与数据许可

> **本章要解决的问题**：一个人做量化，要不要注册公司？注册成什么形式？开哪个账户、做哪些报备？花钱买的数据，能不能用在实盘、能不能贴到对外报告里？一年到底要花多少固定成本，需要多少本金才养得起？

## 本章导读

“一个人做量化”听起来自由，但它仍然是一个经营主体：要收钱、要付钱、要开发票、要报税、要对数据负责。很多人把精力全放在策略上，结果卡在三个地方：账户权限不够、数据许可越界、成本没算清。

本章按“**主体 → 账户与权限 → 数据许可 → 留痕与保密 → 成本账**”五步，把一人量化公司的最小合规框架搭起来。

> **声明**：本章涉及的法律、税务、交易所参数均为**教学示例，非专业意见**。真实办理前，请以现行法律法规、交易所规则和会计师事务所/律师的意见为准。

### 学习目标

- 给定自有资金、收入预期和风险承受度，能列出个人、个体户、有限责任公司三种主体的责任与成本差异，并写出选择理由。
- 给定交易品种清单（股票/两融/期货/期权/程序化），能写出一份“账户 + 权限 + 报备”清单，并标出哪些需要临柜或额外材料。
- 给定一份数据供应商合同，能从“用途、范围、再分发、衍生数据”四问判断许可是否覆盖自己的使用场景。
- 给定启动支出和年度支出，能算出一人公司的启动成本、年度固定成本和盈亏平衡所需收益率。
- 给定券商费率表，能算出一笔股票交易和一笔期货交易的实际费用率。
- 给定业务规模，能判断“注册公司”与“个人身份”哪个更划算，并说明结论在什么条件下反转。

### 先修

第 00 章的环境准备；会 Python 与 pandas 的基础操作；了解期货合约乘数与保证金概念（第 09、42 章会细讲）；对“发票、税前利润”等会计名词有常识即可。

## 在量化研究中的位置

- **谁用**：全职或半全职的个人量化交易者、准备把研究变成产品或资管业务的人、给家人朋友管钱的人。
- **输入**：自有资金、预期收益、交易品种、数据需求、外包需求、所在地政策。
- **输出**：主体选择、账户与权限清单、数据许可判断、合规留痕清单、启动/年度成本表、盈亏平衡点。
- **在流程的哪一步**：在“策略能赚钱”之后、“真金白银上规模”之前；它决定你以什么身份签合同、承担多大责任。
- **失败模式**：用个人账户收管理费、数据只买了个人版却对外展示、程序化交易不报备、成本漏算导致“策略赚钱、账户亏钱”。
- **最小工作流**：算成本 → 选主体 → 开账户/报备 → 核数据许可 → 建留痕 → 每年复核一次。

### 环境准备



In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 120)
print("numpy", np.__version__, "| pandas", pd.__version__)
print("本章参数均为教学示例，非专业意见")


numpy 2.4.4 | pandas 3.0.2
本章参数均为教学示例，非专业意见



## 1. 主体形式：先想清楚“谁承担风险”

### 1.1 三种常见形式

不是所有一人量化都要注册公司。先看三种最常被选择的形式：

| 形式 | 责任边界 | 税（教学示例） | 能开对公户/签合同 | 年维护成本（示例） | 适合 |
|---|---|---|---|---|---|
| 个人身份 | 无限责任，个人财产兜底 | 按个人所得申报 | 不能 | 0 | 自有资金、小规模试验 |
| 个体工商户/个人独资 | 无限责任 | 经营所得 | 一般可开对公户 | 数千元 | 小规模、收入稳定 |
| 有限责任公司 | 以出资为限 | 企业所得税 + 分红个税 | 可以 | 万元级 | 要签合同、要募资、要隔离风险 |

关键差异不在“哪个税低”，而在**责任**和**能不能以机构身份签约**。资管、投顾、数据商用授权、券商机构户，通常都要求一个可签约的主体。

### 1.2 判断顺序

按下面四问走，比直接问“注册公司好不好”更有效：

1. **是否替他人管钱？** 是 → 需要更严格的主体与资质判断，先咨询专业人士。
2. **是否需要对外收费/开票？** 是 → 至少需要能开票的主体。
3. **最大可能亏损是否超过个人可承受范围？** 是 → 需要考虑有限责任隔离。
4. **收入规模是否覆盖年维护成本？** 否 → 先以个人身份小规模运行，别为“看起来正规”先烧钱。

### 1.3 启动成本

把“一次性支出”和“每年重复支出”分开记。下面是一份教学示例，金额按国内一线城市、小微规模估算：



In [2]:

startup = pd.DataFrame(
    [
        ("主体注册（代办+刻章+银行）", 2600, "一次性"),
        ("法务/合同咨询（小时计费）", 3000, "一次性"),
        ("交易设备（电脑+显示器+网络）", 15000, "一次性"),
        ("首年记账报税", 4800, "第一年"),
        ("首年行情/数据（个人研究版）", 12000, "第一年"),
        ("软件与开发工具", 2400, "第一年"),
    ],
    columns=["项目", "金额", "性质"],
)
print(startup.to_string(index=False))
print("启动及首年合计:", startup["金额"].sum(), "元")
print("其中一次性:", startup.loc[startup["性质"] == "一次性", "金额"].sum(), "元")


             项目    金额  性质
 主体注册（代办+刻章+银行）  2600 一次性
  法务/合同咨询（小时计费）  3000 一次性
交易设备（电脑+显示器+网络） 15000 一次性
         首年记账报税  4800 第一年
 首年行情/数据（个人研究版） 12000 第一年
        软件与开发工具  2400 第一年
启动及首年合计: 39800 元
其中一次性: 20600 元



**读数与应用**：这份清单合计 3.98 万元，其中主体注册只有 2,600 元，大头是设备 1.5 万元和首年数据 1.2 万元。**一人公司最容易低估的是数据与设备，而不是注册费。** 常见错误：只算注册费、把设备当沉没成本不计、忘记首年记账费用。

## 2. 账户与交易权限

### 2.1 开户不是“开一个账户”就完了

一人量化至少要分清四类账户：

| 账户/权限 | 主要用途 | 常见前置条件（教学示例） |
|---|---|---|
| 证券账户 | 股票、ETF | 身份证、银行卡、风险测评 |
| 两融账户 | 融资融券 | 资产门槛、交易经验、临柜/线上双录 |
| 期货账户 | 商品/金融期货 | 适当性测试、资金门槛、知识测试 |
| 期权权限 | 商品/ETF 期权 | 等级考试、模拟交易经历、资金门槛 |
| 程序化报备 | 自动下单 | 券商/期货公司报备、接口申请 |

具体门槛、材料和流程以开户机构当期规则为准——本章表格只是提示“有这一关”。

### 2.2 权限矩阵

把“我要做什么”翻译成“我需要哪几个权限”，再逐项确认，不要等下单当天才发现没权限：

1. 只做股票多头 → 证券账户即可；
2. 做股指期货对冲 → 期货账户 + 金融期货权限；
3. 卖期权收权利金 → 期权权限 + 卖方保证金；
4. 程序化自动交易 → 程序化报备 + 接口（部分券商还要求策略说明）；
5. 管他人资金 → 先解决资质与合同，不要先写代码。

### 2.3 券商费用测算

费用直接吃掉收益，必须按“品种 + 方向 + 成交额”逐项算。下面是教学示例费率（**非专业意见，实际以券商/交易所当期标准为准**）：



In [3]:

def stock_cost(amount, commission_rate=0.00025, min_commission=5.0, stamp=0.0005, transfer=0.00001):
    """A 股一笔卖出：佣金 + 印花税 + 过户费。教学示例参数。"""
    commission = max(amount * commission_rate, min_commission)
    return {
        "佣金": commission,
        "印花税": amount * stamp,
        "过户费": amount * transfer,
    }


def futures_cost(lots, fee_per_lot=3.0):
    """期货按手收费。教学示例参数。"""
    return lots * fee_per_lot


amount = 100_000
buy = stock_cost(amount)
sell = stock_cost(amount)
round_trip = sum(buy.values()) + sum(sell.values())
print("股票单边费用", {k: round(v, 2) for k, v in sell.items()})
print("一买一卖合计", round(round_trip, 2), "元，成本率", round(round_trip / amount * 1e4, 2), "个基点")
print("期货 10 手单边费用", futures_cost(10), "元")


股票单边费用 {'佣金': 25.0, '印花税': 50.0, '过户费': 1.0}
一买一卖合计 152.0 元，成本率 15.2 个基点
期货 10 手单边费用 30.0 元



**读数与应用**：10 万元股票一买一卖合计 152 元、约 15.2 个基点（1 bp = 万分之一）；期货按手收费，与成交额无关。**高频策略必须把费用率折算成“每个 bp 收益要打多少折扣”**，否则回测里的 5 bp 毛利可能被成本吃光。常见错误：只算佣金不算印花税、把期货手续费按成交额估算、忘记最低佣金。

### 2.4 从开户到实盘的时间线

一个人办完所有账户，通常要 2–6 周，不是一天。把顺序排好，可以少跑冤枉路：

1. 第 1 周：先开证券账户（多数可线上），同时预约期货账户的适当性测试；
2. 第 2–3 周：完成期货/期权权限与程序化报备；不同公司要求不同，先问清材料清单；
3. 第 4 周：开通数据 API、配置交易接口；用模拟盘跑通“下单—成交—对账”闭环；
4. 第 5–6 周：小额实盘，确认费用、滑点、权限与预期一致，再逐步加资金。

这条时间线的意义是：**在账户没齐之前，不要用“个人账户 + 手工下单”先跑实盘**，否则策略表现和成本结构都不可比。计划里要给开户留出缓冲，不要等策略上线前一天才去申请权限。

## 3. 数据许可：买的是“使用权”，不是“文件”

### 3.1 许可四问

数据合同动辄几十页，抓住四个问题就能判断能不能用：

1. **用途**：只允许“内部研究”，还是允许“实盘交易/资管产品”？
2. **范围**：允许几个人用、几台机器、是否允许给外包/实习生？
3. **再分发**：能否把原始数据、加工后数据给客户或公开？
4. **衍生数据**：用数据算出的因子、信号、指数，归谁、能否商用？

很多“个人研究版”合同只覆盖第 1 条中的“内部研究”，一旦你把回测曲线贴到公众号、把因子做成产品卖，就超出许可。**这不是技术问题，是合同问题。**

### 3.2 个人版与商用版

| 维度 | 个人研究版（示例） | 商用版（示例） |
|---|---|---|
| 年费量级 | 1 万级 | 5 万–10 万级 |
| 用途 | 内部研究、个人账户 | 可含资管/产品/对外展示 |
| 再分发 | 禁止 | 需单独约定 |
| 历史深度/频率 | 可能受限 | 更完整 |
| 典型坑 | 用来跑实盘、贴公开报告 | 仍禁止转售原始数据 |

### 3.3 数据预算

把“必须买”和“可以替代”分开：交易所公开行情、延迟行情、开源数据可以先用；付费数据优先买**你策略真正依赖、且无法替代**的那一部分。



In [4]:

plans = pd.DataFrame(
    [
        ("开源/免费数据", 0, "仅内部研究", "仅日频，需自己清洗"),
        ("个人研究版", 12000, "内部研究、个人账户", "禁止对外展示与再分发"),
        ("商用版", 60000, "对客展示/产品", "按合同约定范围"),
    ],
    columns=["方案", "年费", "允许场景", "主要限制"],
)
print(plans.to_string(index=False))
gap = 60000 - 12000
print("个人版与商用版年费差:", gap, "元，折合每月", round(gap / 12, 1), "元")
print("若年管理规模 100 万，价差占", round(gap / 1_000_000 * 100, 2), "%")
print("对外展示/收费前必须先确认许可，不能先用个人版跑规模")


     方案    年费      允许场景       主要限制
开源/免费数据     0     仅内部研究  仅日频，需自己清洗
  个人研究版 12000 内部研究、个人账户 禁止对外展示与再分发
    商用版 60000   对客展示/产品    按合同约定范围
个人版与商用版年费差: 48000 元，折合每月 4000.0 元
若年管理规模 100 万，价差占 4.8 %
对外展示/收费前必须先确认许可，不能先用个人版跑规模



**读数与应用**：只做内部研究时个人版 1.2 万元/年够用；一旦要对外展示或收费，就必须用商用版（示例 6 万元/年），两者年费差 4.8 万元。**商用版买的不是数据本身，而是许可覆盖的场景。**常见错误：先买个人版跑实盘、以为“改一改数据”就不算再分发、把供应商 API 数据直接存进公开仓库。

## 4. 合规留痕与外包保密

### 4.1 留痕清单

一人公司没有“合规部”，留痕靠习惯。最小清单：

| 留痕对象 | 最小要求 | 为什么 |
|---|---|---|
| 策略决策 | 版本号、参数、变更原因 | 事后能解释“为什么改” |
| 订单与成交 | 时间、价格、数量、来源 | 对账与争议 |
| 通信 | 关键决策的邮件/聊天记录 | 证明指令来源 |
| 数据 | 快照 hash、许可编号 | 证明数据来源合法 |
| 外包 | 合同、NDA、交付物清单 | 界定责任与保密 |

### 4.2 外包与保密

常见外包是“写代码/做数据清洗”。三条底线：

1. 先签 NDA，再给数据；
2. 给最小权限（只给脱敏样本，不给全量）；
3. 交付物验收后收回访问权限，并保留交付清单。

代码托管也要注意：公开仓库默认等于公开策略；私有仓库要开双因素认证，密钥不要写进代码。

### 4.3 一页纸检查表

把上面的内容压缩成一张可勾选的表，每季度过一遍：



In [5]:

checklist = pd.DataFrame(
    [
        ("主体", "经营范围是否覆盖实际业务", "是", ""),
        ("账户", "股票/期货/期权权限是否齐全", "是", ""),
        ("报备", "程序化交易是否已报备", "待办", "本月内"),
        ("数据", "许可是否覆盖实盘与对外展示", "待确认", "查合同第 3 条"),
        ("留痕", "决策/订单/数据快照是否可追溯", "是", ""),
        ("外包", "NDA 与访问权限是否已收回", "待办", "下月"),
    ],
    columns=["类别", "检查项", "状态", "备注"],
)
print(checklist.to_string(index=False))
open_items = checklist["状态"].isin(["待办", "待确认"]).sum()
print("未完成项:", int(open_items), "/", len(checklist))


类别             检查项  状态       备注
主体    经营范围是否覆盖实际业务   是         
账户  股票/期货/期权权限是否齐全   是         
报备      程序化交易是否已报备  待办      本月内
数据   许可是否覆盖实盘与对外展示 待确认 查合同第 3 条
留痕 决策/订单/数据快照是否可追溯   是         
外包  NDA 与访问权限是否已收回  待办       下月
未完成项: 3 / 6



**读数与应用**：6 项里有 3 项未完成，说明“个人能跑通”不等于“公司能合规运营”。常见错误：把检查表当一次性任务、外包结束后不收回权限、公开仓库里放配置文件。

### 4.3 留痕工具的最小实现

不需要一上来就买合规系统。一个人可以用三个文件起步：

1. `decisions.md`：每次改参数/换策略，写日期、改动、原因、预期影响；
2. `trades.csv`：逐笔成交导出，按月归档，文件名带日期；
3. `data_license.md`：数据合同编号、许可范围、到期日、续费提醒。

三个文件放进私有 Git 仓库，提交信息写清“改了什么”。这套土办法不能替代审计，但能在被问“为什么这么改”时，三分钟内拿出记录。

## 5. 成本账与盈亏平衡

### 5.1 年度固定成本

一人公司的年度固定成本，至少要包含：数据、服务器、记账报税、软件、保险、交易通道与合规咨询。下面是一份教学示例：



In [6]:

annual = pd.DataFrame(
    [
        ("数据（个人研究版）", 12000),
        ("云服务器/存储", 4800),
        ("记账报税", 4800),
        ("软件与工具", 2400),
        ("合规/法务咨询（按年摊销）", 6000),
        ("商业保险", 1200),
    ],
    columns=["项目", "年度金额"],
)
print(annual.to_string(index=False))
annual_total = annual["年度金额"].sum()
capital = 500_000
print("年度固定成本合计:", annual_total, "元")
print("占自有资金比例:", round(annual_total / capital * 100, 2), "%")
print("覆盖成本所需年化收益（不计税费）:", round(annual_total / capital * 100, 2), "%")


           项目  年度金额
    数据（个人研究版） 12000
      云服务器/存储  4800
         记账报税  4800
        软件与工具  2400
合规/法务咨询（按年摊销）  6000
         商业保险  1200
年度固定成本合计: 31200 元
占自有资金比例: 6.24 %
覆盖成本所需年化收益（不计税费）: 6.24 %



**读数与应用**：年成本 3.12 万元、自有资金 50 万元，意味着策略必须先赚 6.24% 才“白干”；若再算个人所得税、滑点和融资成本，真实门槛更高。**一人量化最常见的死法不是爆仓，而是收益覆盖不了固定成本。** 常见错误：不把自己的人工成本算进去、按牛市收益算常态、忽略数据涨价。

## 本章小结

### 一张表

| 环节 | 关键动作 | 最容易踩的坑 |
|---|---|---|
| 主体 | 按责任/签约/税三问选择 | 为“正规”先烧钱 |
| 账户 | 权限清单逐项确认 | 下单当天才发现没权限 |
| 数据 | 用途/范围/再分发/衍生四问 | 个人版用于商用场景 |
| 留痕 | 决策/订单/数据/通信 | 事后无法解释 |
| 成本 | 启动 + 年度分开记 | 漏算数据与设备 |
| 盈亏平衡 | 年成本 / 自有资金 | 用牛市收益当常态 |

### 八句话

1. 主体先看责任，再看税。
2. 数据买的是许可，不是文件。
3. 个人研究版不等于实盘版。
4. 程序化交易要报备。
5. 留痕是为了事后解释得清。
6. 外包先签 NDA 再给数据。
7. 固定成本决定最低收益门槛。
8. 本章所有参数都是教学示例，办理前找专业人士。

## 本章习题

**1.** 个人、个体户、有限责任公司，主要区别是哪三件事？

**2.** 自有资金 30 万元，年度固定成本 3.6 万元，覆盖成本所需年化收益率是多少？

**3.** 10 万元股票卖出，佣金万 2.5（最低 5 元）、印花税 0.05%、过户费 0.001%，卖出费用多少？

**4.** 数据合同里“用途、范围、再分发、衍生”分别要问什么？

**5.** 程序化交易不做报备，可能遇到什么后果？

**6.** 外包开发时，三条保密底线是什么？

**7.** 为什么“策略年化 15%”不一定能养活一人公司？

**8.** 什么情况下应该先以个人身份运行，而不是立刻注册公司？

<details><summary>参考答案</summary>

**1.** 责任边界（是否以个人财产兜底）、能否以机构身份签约/开票、税负与维护成本。

**2.** 3.6 / 30 = 12%。

**3.** 佣金 max(100000×0.00025, 5)=25 元；印花税 50 元；过户费 1 元；合计 76 元。

**4.** 用途：内部研究还是实盘/产品；范围：几个人几台机器、能否给外包；再分发：能否给客户或公开；衍生：因子/信号/指数归谁、能否商用。

**5.** 可能被限制接口、暂停程序化权限，甚至影响账户正常使用；具体后果以券商与监管规则为准（教学示例，非专业意见）。

**6.** 先签 NDA；只给脱敏样本与最小权限；交付后收回权限并保留清单。

**7.** 还要扣数据/设备/记账等固定成本、交易成本、税费；15% 是毛利口径时，净收益可能低于成本门槛。

**8.** 收入规模覆盖不了年维护成本、只管理自有小额资金、暂时不需要对外签约/开票时。

</details>

---

## 参考资料

- 中国证监会、各期货交易所投资者适当性管理办法（以最新发布为准）。
- 各券商/期货公司开户与程序化交易报备指引（以开户机构当期要求为准）。
- 数据供应商合同范本中的“许可范围/再分发/衍生数据”条款。
- 第 00 章（环境准备）、第 09 章（期限结构与换月）、第 42 章（商品期权合约与保证金）。
